# H5 To Parameters

Extract posterior parameter samples from an `emcee` HDF5 backend, convert them to full physical parameters, and save them to CSV.

In [1]:
# Top-level configuration
fit_name = "Final_old"
input_h5_path = "source_data/emcee_gated_global_9d.h5"
output_csv_path = "bayesian parameters/bayesian_parameters_3000.csv"

burn = 1000
thin = 10
final_parameter_rows = 3000  # set to None to keep all retained samples
rng_seed = 123

include_log_prob = True

In [2]:
from julia.api import Julia
from julia import Main

import re
from pathlib import Path

import emcee
import numpy as np
import pandas as pd
from IPython.display import display

In [3]:
TMD_fitting_root = Path("../")

def include(name):
    path = (TMD_fitting_root / name).resolve()
    Main.eval(f'include(raw"{path.as_posix()}")')

include(f"Cards/{fit_name}.jl")

initial_params = np.asarray(Main.initial_params, dtype=float)
bounds = Main.eval("bounds_raw")
LOWER = np.array([b[0] for b in bounds], dtype=float)
UPPER = np.array([b[1] for b in bounds], dtype=float)

theta_center_phys = initial_params.copy()
theta_center_norm = np.clip((theta_center_phys - LOWER) / (UPPER - LOWER), 0.0, 1.0).astype(np.float32)

frozen_julia = np.asarray(Main.eval("frozen_indices"), dtype=int)
frozen_py = frozen_julia - 1 if frozen_julia.size else np.asarray([], dtype=int)
frozen_vals_norm = theta_center_norm[frozen_py].copy() if frozen_py.size else np.empty((0,), dtype=np.float32)

def apply_freeze(th):
    arr = np.asarray(th, dtype=np.float32)
    one_d = (arr.ndim == 1)
    if one_d:
        arr = arr[None, :]
    if frozen_py.size:
        arr[:, frozen_py] = frozen_vals_norm[None, :]
    return arr[0] if one_d else arr

mask = np.ones(len(theta_center_norm), dtype=bool)
if frozen_py.size:
    mask[frozen_py] = False
free_idx_replica = np.where(mask)[0]
xinit_replica = apply_freeze(theta_center_norm.copy())

def free_to_full(xfree, free_idx, xinit):
    x = np.asarray(xinit, dtype=np.float32).copy()
    x[np.asarray(free_idx, dtype=int)] = np.asarray(xfree, dtype=np.float32)
    x = np.clip(x, 0.0, 1.0)
    x = apply_freeze(x)
    return x.astype(np.float32)

def norm_to_phys(th_norm):
    th = np.asarray(th_norm, dtype=np.float32)
    return LOWER + th * (UPPER - LOWER)

card_path = TMD_fitting_root / "Cards" / f"{fit_name}.jl"
card_text = card_path.read_text(encoding="utf-8")
struct_match = re.search(r"struct\s+Params_Struct(.*?)end", card_text, re.S)
if struct_match is None:
    raise ValueError(f"Could not find Params_Struct in {card_path}")

param_names = re.findall(r"([A-Za-z_][A-Za-z0-9_]*)\s*::\s*Float32", struct_match.group(1))
if len(param_names) != len(initial_params):
    raise ValueError(
        f"Params_Struct size ({len(param_names)}) does not match initial_params size ({len(initial_params)})"
    )

param_info_df = pd.DataFrame(
    {
        "index": np.arange(len(param_names), dtype=int),
        "parameter": param_names,
        "initial_value": initial_params,
        "frozen": np.isin(np.arange(len(param_names), dtype=int), frozen_py),
    }
)

print("Free parameter count:", len(free_idx_replica))
display(param_info_df)

Free parameter count: 9


,index,parameter,initial_value,frozen
0,0,lambda1,0.023656,False
1,1,lambda2,1.054291,False
2,2,lambda3,-2.354365,False
3,3,logx0,-5.207703,False
4,4,sigx,1.103274,False
5,5,amp,-0.431106,False
6,6,BNP,1.494665,False
7,7,c0,0.070013,False
8,8,c1,0.027637,False


In [4]:
input_h5_path = Path(input_h5_path)
output_csv_path = Path(output_csv_path)

backend_candidates = [input_h5_path]
if not input_h5_path.is_absolute() and not input_h5_path.exists():
    backend_candidates.append((Path.cwd() / input_h5_path).resolve())

for candidate in backend_candidates:
    if candidate.exists():
        input_h5_path = candidate
        break
else:
    raise FileNotFoundError(
        f"Could not resolve input_h5_path from candidates: {[str(p) for p in backend_candidates]}"
    )

if final_parameter_rows is not None and int(final_parameter_rows) <= 0:
    raise ValueError("final_parameter_rows must be positive or None")

backend = emcee.backends.HDFBackend(str(input_h5_path), read_only=True)
chain_free = backend.get_chain(discard=burn, flat=True, thin=thin).astype(np.float32)
logp_flat = backend.get_log_prob(discard=burn, flat=True, thin=thin).astype(np.float64)
source_index = np.arange(len(chain_free), dtype=int)

if chain_free.ndim != 2:
    raise ValueError(f"Expected flattened chain to be 2D, got shape {chain_free.shape}")
if chain_free.shape[1] != len(free_idx_replica):
    raise ValueError(
        f"Backend ndim ({chain_free.shape[1]}) does not match free parameter count ({len(free_idx_replica)})"
    )

rng = np.random.default_rng(rng_seed)
n_retained_before_cap = len(chain_free)
if final_parameter_rows is not None and len(chain_free) > int(final_parameter_rows):
    keep_idx = np.sort(rng.choice(len(chain_free), size=int(final_parameter_rows), replace=False))
    chain_free = chain_free[keep_idx]
    logp_flat = logp_flat[keep_idx]
    source_index = source_index[keep_idx]

chain_full_norm = np.stack(
    [free_to_full(x, free_idx_replica, xinit_replica) for x in chain_free],
    axis=0,
).astype(np.float32)
chain_full_phys = norm_to_phys(chain_full_norm).astype(np.float64)

parameters_df = pd.DataFrame(chain_full_phys, columns=param_names)
parameters_df.insert(0, "source_index", source_index)
if include_log_prob:
    parameters_df.insert(1, "log_prob", logp_flat)

print("Resolved backend:", input_h5_path)
print("Retained rows before final cap:", n_retained_before_cap)
print("Rows written:", len(parameters_df))
display(parameters_df.head())

Resolved backend: source_data\emcee_gated_global_9d.h5
Retained rows before final cap: 51200
Rows written: 3000


,source_index,log_prob,lambda1,lambda2,lambda3,logx0,sigx,amp,BNP,c0,c1
0,9,-186.607928,0.004001,1.066987,-2.639572,-5.354135,1.032392,-0.464033,1.915794,0.059724,0.022447
1,31,-186.622503,0.004191,1.143977,-2.681464,-5.042142,1.527820,-0.423431,1.922110,0.062913,0.023972
2,41,-184.315792,0.002548,1.122492,-2.657517,-5.301107,1.660783,-0.429345,1.741697,0.064500,0.026654
3,51,-185.509553,-0.008229,1.084817,-2.818570,-5.745763,1.447606,-0.570028,1.801397,0.065770,0.027093
4,77,-185.064431,0.001206,1.091165,-2.826395,-5.566091,1.262800,-0.494137,1.995701,0.063111,0.024153


In [5]:
output_csv_path.parent.mkdir(parents=True, exist_ok=True)
parameters_df.to_csv(output_csv_path, index=False)

print(f"Saved {len(parameters_df)} rows to {output_csv_path.resolve()}")

Saved 3000 rows to C:\Users\congyue zhang\Desktop\OpenCL fitter\TMD-Fits-Minimal\Bayesian Analysis\bayesian parameters\bayesian_parameters_3000.csv
